In [15]:
import torch
import math

class TokenEmbeddings:
    def __init__(self, vocab_size, num_dims):
        self.weights = torch.randn(vocab_size, num_dims)

    def forward(self, indices):
        self.indices = indices
        return self.weights[indices]

    def backward(self, grad_out):
        
        if self.weights.grad is None:
            self.weights.grad = torch.zeros_like(self.weights)
            
        flat_indices = self.indices.flatten()
        flat_grad = grad_out.flatten(0, -2)
        
        for i, token_id in enumerate(flat_indices):
            self.weights.grad[token_id] += flat_grad[i]
            
        return None

    def parameters(self):
        return [self.weights]


class PositionalEmbeddings:
    
    def __init__(self, max_seq_len, num_dims):
        self.weights = torch.randn(max_seq_len, num_dims)

    def forward(self, positions):
        self.positions = positions
        return self.weights[positions]

    def backward(self, grad_out):
        
        if self.weights.grad is None:
            self.weights.grad = torch.zeros_like(self.weights)

        grad_sum = grad_out.sum(dim=0)
        flat_positions = self.positions.flatten()
        
        self.weights.grad.index_add_(0, flat_positions, grad_sum)
        return None

    def parameters(self):
        return [self.weights]



class LinearLayer:
    def __init__(self,in_features,out_features,bias=True):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.has_bias=bias
            
        if bias:
            self.bias = torch.zeros(out_features)
        else:
            self.bias=None

    def forward(self,x):
        self.x = x
        out = x @ self.weights.T
        
        if self.has_bias:
            out=out+self.bias
            
        return out

    def backward(self,grad_out):

        x_shape = self.x.shape
        grad_flat = grad_out.flatten(0,-2)
        x_flat = self.x.flatten(0,-2)
        
        grad_inputs = grad_flat @ self.weights
        self.weights.grad = grad_flat.T @ x_flat

        if self.has_bias:
            self.bias.grad = grad_flat.sum(dim=0)

        return grad_inputs.reshape(x_shape)


class Relu:
    def forward(self,x):
        self.x = x
        return torch.clamp(x,min=0);

    def backward(self,grad_out):
        return grad_out * (self.x > 0)


class Dropout:
    def __init__(self, p=0.1, training=True):
        self.p = p
        self.training = training
        self.mask = None
        
    def forward(self, x):
        self.x = x
        if self.training:
            self.mask = (torch.rand_like(x) > self.p).float() / (1 - self.p)
            return x * self.mask
        else:
            return x

    def backward(self, grad_out):
        if self.training:
            return grad_out * self.mask
        else:
            return grad_out

    
class softmax:
    def forward(self, scores):
        max_score = torch.max(scores, dim=-1, keepdim=True).values
        scores_exp = torch.exp(scores - max_score)
        self.scores_sum = scores_exp.sum(dim=-1, keepdim=True)
        self.out = scores_exp / self.scores_sum
        return self.out

    def backward(self, grad_out):
        sum_term = (grad_out * self.out).sum(dim=-1, keepdim=True)
        grad_scores = self.out * (grad_out - sum_term)
        return grad_scores



class CausalSelfAttention:

    def __init__(self, num_dims, num_heads):  
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.head_dims = num_dims // num_heads

        self.w_q = LinearLayer(num_dims,num_dims,bias=False)
        self.w_k = LinearLayer(num_dims,num_dims,bias=False)
        self.w_v = LinearLayer(num_dims,num_dims,bias=False)

        self.proj_out = LinearLayer(num_dims,num_dims,bias=True)
        self.softmax = softmax()

        self.attn_drop = Dropout(p=0.1,training=True)
        self.resid_drop = Dropout(p=0.1,training=True)

    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q.forward(x)
        K = self.w_k.forward(x)
        V = self.w_v.forward(x)

        Q = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        self.Q = Q
        self.K = K
        self.V = V
        
        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T,T,dtype=torch.bool,device=scores.device),diagonal=1)

        scores = scores.masked_fill(masks,float("-inf"))
        
        self.attn_scores = self.softmax.forward(scores)
        self.attn_scores = self.attn_drop.forward(self.attn_scores)
        self.out = self.attn_scores @ self.V
        self.out = self.out.transpose(1,2).contiguous().view(B,T,D)
        self.out = self.proj_out.forward(self.out)
        self.out = self.resid_drop.forward(self.out)
        return self.out


    def backward(self,grad_out):
        B,T,D = grad_out.shape
        grad_resid = self.resid_drop.backward(grad_out)
        grad_proj = self.proj_out.backward(grad_resid)
        grad_proj = grad_proj.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        grad_attn = grad_proj @ self.V.transpose(-2,-1)
        grad_v = self.attn_scores.transpose(-2,-1) @ grad_proj

        grad_probs = self.attn_drop.backward(grad_attn)
        grad_scores = self.softmax.backward(grad_probs)   

        scale = 1.0 / math.sqrt(self.head_dims)
        grad_scores = grad_scores * scale
        
        grad_q = grad_scores @ self.K
        grad_k = grad_scores.transpose(-2,-1) @ self.Q
        

        grad_Q = grad_q.transpose(1, 2).contiguous().view(B, T, D)
        grad_K = grad_k.transpose(1, 2).contiguous().view(B, T, D)
        grad_V = grad_v.transpose(1, 2).contiguous().view(B, T, D)

        grad_x_q = self.w_q.backward(grad_Q)
        grad_x_k = self.w_k.backward(grad_K)
        grad_x_v = self.w_v.backward(grad_V)

        grad_x = grad_x_q+grad_x_k+grad_x_v

        return grad_x
        

class LayerNorm:
    def __init__(self, num_dims, eps=1e-16):
        self.gamma = torch.ones(num_dims)
        self.beta = torch.zeros(num_dims)
        self.eps = eps
        self.num_dims = num_dims

    def forward(self, x):
        self.x = x
        self.avg = torch.mean(x, dim=-1, keepdim=True)
    
        x_centered = x - self.avg
        self.var = torch.mean(x_centered**2, dim=-1, keepdim=True)
        
        inv_sum = 1.0 / torch.sqrt(self.var + self.eps)
        self.x_norm = x_centered * inv_sum

        return self.x_norm * self.gamma + self.beta

    def backward(self, grad_out):
        if self.beta.grad is None:
            self.beta.grad = torch.zeros_like(self.beta)
            self.gamma.grad = torch.zeros_like(self.gamma)

        self.beta.grad += torch.sum(grad_out, dim=(0, 1))
        self.gamma.grad += torch.sum(grad_out * self.x_norm, dim=(0, 1)) # Note: gamma grad uses x_norm!

        grad_x_norm = grad_out * self.gamma

        return (1.0 / torch.sqrt(self.var + self.eps)) * (
            grad_x_norm
            - torch.mean(grad_x_norm, dim=-1, keepdim=True)
            - self.x_norm * torch.mean(grad_x_norm * self.x_norm, dim=-1, keepdim=True)
        )


class MyMlp:
    
    def __init__(self, in_features, hidden_features, out_features):
        self.linear1 = LinearLayer(in_features, hidden_features)
        self.relu = Relu()
        self.linear2 = LinearLayer(hidden_features, out_features)
        self.dropout = Dropout(p=0.1, training=True)

    def forward(self, x):
        x = self.linear1.forward(x)
        x = self.relu.forward(x)
        x = self.linear2.forward(x)
        x = self.dropout.forward(x)
        return x

    def backward(self, grad_out):
        grad_drop = self.dropout.backward(grad_out)
        grad_linear2 = self.linear2.backward(grad_drop)
        grad_relu = self.relu.backward(grad_linear2)
        grad_linear1 = self.linear1.backward(grad_relu)
        return grad_linear1



class MyMseLoss:

    def forward(self,y_pred,y_true):
        self.y_pred = y_pred
        self.y_true = y_true

        loss = torch.mean((y_pred - y_true)**2)
        return loss

    def backward(self,grad_out):
        N = self.y_pred.numel()
        out = grad_out *  (2/N)* (self.y_pred - self.y_true)
   
        return out


class Optim:
    def __init__(self, params, lr=0.001):
        self.params = list(params)
        self.lr = lr

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad
    

class Transformer:

    def __init__(self, num_dims, num_heads):
        self.ln_1 = LayerNorm(num_dims)
        self.attn = CausalSelfAttention(num_dims, num_heads)
        self.ln_2 = LayerNorm(num_dims)
        self.mlp = MyMlp(in_features=num_dims, hidden_features=4 * num_dims, out_features=num_dims)
        
    def forward(self, x):
        norm1 = self.ln_1.forward(x)
        attn_out = self.attn.forward(norm1)
        x1 = x + attn_out

        norm2 = self.ln_2.forward(x1)
        mlp_out = self.mlp.forward(norm2)
        x2 = x1 + mlp_out

        return x2

    def backward(self, grad_out):
        
        grad_mlp = self.mlp.backward(grad_out)
        grad_norm2 = self.ln_2.backward(grad_mlp)
        grad_x1 = grad_out + grad_norm2

        grad_attn = self.attn.backward(grad_x1)
        grad_norm1 = self.ln_1.backward(grad_attn)
        grad_x = grad_x1 + grad_norm1

        return grad_x

class GPT:
    def __init__(self,num_dims,num_heads,num_layers,vocab_size,seq_len):
        self.seq_len = seq_len
        self.wte = TokenEmbeddings(vocab_size,num_dims)
        self.wpe = PositionalEmbeddings(seq_len,num_dims)

        self.blocks = [Transformer(num_dims,num_heads) for _ in range(num_layers)]
        
        self.ln_f = LayerNorm(num_dims)
        self.lm_head = LinearLayer(num_dims,vocab_size)

    def forward(self,idx):
        B,T = idx.shape

        pos = torch.arange(T, device=idx.device)

        tok_emb = self.wte.forward(idx)
        pos_emb = self.wpe.forward(pos)

        x = tok_emb+pos_emb

        for block in self.blocks:
            x = block.forward(x)

        x  = self.ln_f.forward(x)
        logits = self.lm_head.forward(x)

        return logits


    def backward(self, grad_out):
      
        grad_logits = self.lm_head.backward(grad_out)
        grad_ln = self.ln_f.backward(grad_logits)

        grad = grad_ln
        for block in reversed(self.blocks):
            grad = block.backward(grad)

        self.wpe.backward(grad)
        self.wte.backward(grad)

        return grad          
    

vocab_size = 50257
seq_len = 64
num_dims = 128
num_heads = 4
num_layers = 2
B, T = 2, 16

model = GPT( num_dims=num_dims,num_heads=num_heads,num_layers=num_layers,vocab_size=vocab_size,seq_len=seq_len)

dummy_idx = torch.randint(0, vocab_size, (B, T))
logits = model.forward(dummy_idx)
print(f"Logits shape: {logits.shape}")  

dummy_grad = torch.randn_like(logits)
grad_x = model.backward(dummy_grad)
print(f"Grad_x shape: {grad_x.shape}")




Logits shape: torch.Size([2, 16, 50257])
Grad_x shape: torch.Size([2, 16, 128])


### Note to Readers / Future Me

I did not write manual `parameters()` methods for all sub-modules. I was exhausted. 😭

The entire forward pass, backward pass, chain rule, attention math and residual routing are fully derived and verified end-to-end.

Manually wiring a recursive list collector for weight pointers is pure boilerplate busywork so PyTorch can take the wheel for parameter registration. But that's what makes this code human. 😂
